# Clinical Data Lakehouse Modernization Pipeline

This notebook orchestrates the end-to-end medallion architecture pipeline for clinical data using Databricks. The pipeline consists of the following steps:

1. **Bronze Layer**: Ingests raw data into the bronze table and storage.
2. **Silver Layer**: Cleanses and transforms data into the silver layer.
3. **Gold Layer**: Aggregates and prepares data for analytics in the gold layer.
4. **Audit Layer**: Runs audit and orchestration checks.
5. **Cleanup**: Truncates the bronze table and removes bronze files from S3 after successful pipeline execution.
6. **Verification**: Confirms that the bronze folder is empty.

This notebook ensures a clean, repeatable, and auditable data pipeline for clinical data modernization.

##### importing the required libraires

In [0]:
%run ./config

In [0]:
from datetime import datetime

# Notebook paths for orchestration
BRONZE_NOTEBOOK = (
    "/Workspace/Users/takiakio09@gmail.com/"
    "Clinical_DataLakehouse_Modernization/"
    "medallion architecture/01_Bronze_Load"
)

SILVER_NOTEBOOK = (
    "/Workspace/Users/takiakio09@gmail.com/"
    "Clinical_DataLakehouse_Modernization/"
    "medallion architecture/02_Silver_Load"
)

GOLD_NOTEBOOK = (
    "/Workspace/Users/takiakio09@gmail.com/"
    "Clinical_DataLakehouse_Modernization/"
    "medallion architecture/03_Gold_Load"
)

AUDIT_NOTEBOOK = (
    "/Workspace/Users/takiakio09@gmail.com/"
    "Clinical_DataLakehouse_Modernization/"
    "medallion architecture/04_Audit_And_Orchestration"
)

# Table and path references from config
BRONZE_TABLE = cfg['bronze_table']
BRONZE_PATH = cfg['bronze_path']

##### Run pipeline

In [0]:
%skip
try:

    start_time = datetime.now()

    print("Running Bronze Layer...")

    dbutils.notebook.run(
        BRONZE_NOTEBOOK,
        0
    )

    print("Bronze Completed")


    print("Running Silver Layer...")

    dbutils.notebook.run(
        SILVER_NOTEBOOK,
        0
    )

    print("Silver Completed")


    print("Running Gold Layer...")

    dbutils.notebook.run(
        GOLD_NOTEBOOK,
        0
    )

    print("Gold Completed")


    print("Running Audit Layer...")

    dbutils.notebook.run(
        AUDIT_NOTEBOOK,
        0
    )

    print("Audit Completed")


    end_time = datetime.now()

    print(
        "Pipeline Completed Successfully"
    )

    print(
        "Execution Time :",
        end_time - start_time
    )

except Exception as e:

    print(
        "Pipeline Failed :",
        str(e)
    )

    raise e

##### Truncate bronze after success

In [0]:
try:

    spark.sql(f"""
        TRUNCATE TABLE {BRONZE_TABLE}
    """)

    print(f"Bronze table {BRONZE_TABLE} truncated successfully")

except Exception as e:

    print("Error :", e)

    raise e

##### Remove bronze files from s3

In [0]:
try:

    dbutils.fs.rm(
        BRONZE_PATH,
        True
    )

    print(f"Bronze S3 files removed successfully from {BRONZE_PATH}")

except Exception as e:

    print("Error :", e)

    raise e

##### Verification

In [0]:
try:

    bronze_base_path = BRONZE_PATH.rsplit('/', 2)[0] + '/'
    
    print(
        dbutils.fs.ls(
            bronze_base_path
        )
    )

except Exception as e:

    print(
        f"Bronze folder is empty: {bronze_base_path}"
    )